# 6장. 신경망의 규제

학습 자료의 손실이 계속 줄어드는데, 언제 멈추고 무엇을 제한해야 하는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 12분쯤 걸린다.

## 6.1 큰 신경망을 오래 학습시키고 검증 자료로 재기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
import copy
import torch
from torch import nn

def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def column(values):
    """넘파이 배열을 (자료의 수, 1) 모양의 32비트 텐서로 바꾼다."""
    return torch.tensor(np.asarray(values)[:, None], dtype=torch.float32)

def pad(text, width):
    """글자 수를 맞춰 오른쪽을 공백으로 채운다. 한글은 두 칸으로 센다."""
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

torch.set_num_threads(1)
days = weekday.loc["2023"].index                 # 2023년 평일 247일의 날짜
week = days.isocalendar().week.to_numpy()        # 그 날이 그해의 몇째 주인지
loss_fn = nn.MSELoss()

def prepare(residue):
    """주 번호를 4로 나눈 나머지가 residue인 주를 검증 자료로 뗀다.
    표준화에 쓰는 평균과 표준편차는 학습 자료로만 정한다."""
    held = week % 4 == residue
    t_mean, t_std = T23[~held].mean(), T23[~held].std()
    y_mean, y_std = Y23[~held].mean(), Y23[~held].std()
    tensors = (column((T23[~held] - t_mean) / t_std),
               column((Y23[~held] - y_mean) / y_std),
               column((T23[held] - t_mean) / t_std),
               column((Y23[held] - y_mean) / y_std),
               column((T25 - t_mean) / t_std))
    return tensors, (t_mean, t_std, y_mean, y_std), held

tensors, (t_mean, t_std, y_mean, y_std), held = prepare(1)
X_tr, Z_tr, X_va, Z_va, X25 = tensors
T_tr, Y_tr, T_va, Y_va = T23[~held], Y23[~held], T23[held], Y23[held]

print(len(T_tr), len(T_va), len(T25), sorted(set(week[held].tolist())))
print(round(T_tr.min(), 1), round(T_tr.max(), 1))
print(round(T_va.min(), 1), round(T_va.max(), 1))
print(round(T25.min(), 1), round(T25.max(), 1))
print(round(Y_tr.std(), 2), round(Y_va.std(), 2))
print(round(loss_fn(torch.zeros_like(Z_va), Z_va).item(), 4))

In [ ]:
def make(seed, width=64, depth=3, drop=0.0):
    """은닉층이 depth개이고 층마다 유닛이 width개인 신경망. drop은 6.4절에서 쓴다."""
    torch.manual_seed(seed)
    layers, m = [], 1
    for _ in range(depth):
        layers += [nn.Linear(m, width), nn.ReLU()]
        if drop > 0:
            layers.append(nn.Dropout(drop))
        m = width
    return nn.Sequential(*layers, nn.Linear(m, 1))

def count(model):
    return sum(param.numel() for param in model.parameters())

def gw(model, x):
    """표준화한 입력을 받아 전력수요(GW)를 예측한다. 평가 모드로 계산한다."""
    model.eval()
    with torch.no_grad():
        return model(x).numpy()[:, 0].astype(np.float64) * y_std + y_mean

print(count(make(0)), count(make(0, width=8, depth=1)))

In [ ]:
def fit(model, optimizer, epochs=1000, seed=0):
    """배치 크기 32로 학습시키고 에폭마다 세 가지를 기록한다.
    열: 학습 자료의 손실, 검증 자료의 손실, 2025년 RMSE(GW).
    검증 손실이 가장 낮았던 에폭의 가중치도 함께 돌려준다."""
    shuffler = torch.Generator().manual_seed(seed)
    log, best, best_state = [], float("inf"), None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_tr), generator=shuffler)
        for start in range(0, len(X_tr), 32):
            rows = order[start:start + 32]
            optimizer.zero_grad()
            loss_fn(model(X_tr[rows]), Z_tr[rows]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(X_tr), Z_tr).item()
            valid_loss = loss_fn(model(X_va), Z_va).item()
        log.append((train_loss, valid_loss, rmse(Y25, gw(model, X25))))
        if valid_loss < best:
            best, best_state = valid_loss, copy.deepcopy(model.state_dict())
    return np.array(log), best_state

def trial(make_model, make_optimizer, epochs=1000):
    """시드 0~4로 학습시켜 기록 5개를 돌려준다."""
    logs = []
    for seed in range(5):
        model = make_model(seed)
        logs.append(fit(model, make_optimizer(model.parameters()), epochs, seed)[0])
    return logs

def show(name, logs):
    """왼쪽: 마지막 에폭의 학습 손실, 검증 손실, 2025년 RMSE(범위).
    오른쪽: 검증 손실이 가장 낮았던 에폭, 그 에폭의 검증 손실, 2025년 RMSE(범위)."""
    last = np.array([log[-1] for log in logs])
    when = np.array([log[:, 1].argmin() for log in logs])
    best = np.array([log[k] for log, k in zip(logs, when)])
    def cells(rows):
        marks = rows[:, 2]
        return (f"{np.median(rows[:, 0]):.3f} {np.median(rows[:, 1]):.3f} "
                f"{np.median(marks):.2f} ({marks.min():.2f}~{marks.max():.2f})")
    middle = f" | {int(np.median(when)) + 1:>4} "
    print(pad(name, 20) + cells(last) + middle + cells(best)[6:])

def adam(rate=0.003, **options):
    return lambda params: torch.optim.Adam(params, lr=rate, **options)

results = {}
results["큰 모델"] = trial(make, adam())
results["작은 모델"] = trial(lambda seed: make(seed, width=8, depth=1), adam())
show("큰 모델", results["큰 모델"])
show("작은 모델", results["작은 모델"])

In [ ]:
model = make(0)                            # 시드 0을 한 번 더 학습시켜 모델을 남긴다
big_log, best_state = fit(model, adam()(model.parameters()))
big_last, big_best = model, make(0)
big_best.load_state_dict(best_state)       # 검증 손실이 가장 낮았던 가중치
small_last = make(0, width=8, depth=1)
fit(small_last, adam()(small_last.parameters()))

for epoch in [1, 4, 5, 10, 20, 50, 100, 300, 1000]:
    train_loss, valid_loss, mark = big_log[epoch - 1]
    print(f"{epoch:>5} {train_loss:.4f} {valid_loss:.4f} {mark:.2f}")
turn = int(big_log[:, 1].argmin())
print(turn + 1, round(big_log[turn, 1], 4), round(big_log[turn, 0], 4))
print(round(big_log[:, 0].min(), 4), int(big_log[:, 0].argmin()) + 1)
print(int((big_log[99:, 1] > big_log[99:, 0]).sum()), len(big_log[99:]))
print(int(big_log[:, 2].argmin()) + 1, round(big_log[:, 2].min(), 2))
small_log = results["작은 모델"][0]
print(round(small_log[99:, 2].min(), 2), round(small_log[99:, 2].max(), 2))

In [ ]:
grid = np.array([-9.4, -5.0, 0.0, 10.0, 20.0, 25.0, 30.0, 33.0])
grid_x = column((grid - t_mean) / t_std)
print("기온        ", " ".join(f"{t:>6.1f}" for t in grid))
for name, net in [("큰 모델, 끝", big_last), ("큰 모델, 4에폭", big_best),
                  ("작은 모델, 끝", small_last)]:
    print(pad(name, 16) + " ".join(f"{v:>6.1f}" for v in gw(net, grid_x)))

fine = np.linspace(T_tr.min(), T_tr.max(), 400)
fine_x = column((fine - t_mean) / t_std)
for name, net in [("큰 모델, 끝", big_last), ("큰 모델, 4에폭", big_best),
                  ("작은 모델, 끝", small_last)]:
    curve = gw(net, fine_x)
    slope = np.diff(curve)
    turns = int((np.sign(slope[1:]) != np.sign(slope[:-1])).sum())
    jump = np.abs(curve[5:] - curve[:-5])          # 다섯 칸은 약 0.5℃다
    print(pad(name, 16) + f"방향 바뀜 {turns:>2}번, 0.5℃ 사이 가장 큰 변화 "
          f"{jump.max():.1f}GW({fine[jump.argmax()]:.1f}℃), "
          f"가장 낮은 곳 {fine[curve.argmin()]:.1f}℃")

## 6.2 조기 종료: 검증 손실이 가장 낮은 가중치 쓰기

In [ ]:
def stop_point(valid_trace, patience):
    """검증 손실이 patience번 연속으로 나아지지 않으면 멈춘다.
    (멈춘 에폭, 멈출 때까지 검증 손실이 가장 낮았던 에폭)을 돌려준다. 에폭은 1부터 센다."""
    best, best_epoch = float("inf"), 0
    for epoch, value in enumerate(valid_trace, start=1):
        if value < best:
            best, best_epoch = value, epoch
        elif epoch - best_epoch >= patience:
            return epoch, best_epoch
    return len(valid_trace), best_epoch

print("인내    멈춘 에폭                 멈춘 때의 RMSE      되돌린 가중치의 RMSE")
for name in ["큰 모델", "작은 모델"]:
    print(name)
    for patience in [5, 20, 50, 200]:
        stops, at_stop, at_best = [], [], []
        for log in results[name]:
            stop, best_epoch = stop_point(log[:, 1], patience)
            stops.append(stop)
            at_stop.append(log[stop - 1, 2])
            at_best.append(log[best_epoch - 1, 2])
        print(f"{patience:>4}    {str(stops):<26}"
              f"{np.median(at_stop):.2f} ({min(at_stop):.2f}~{max(at_stop):.2f})    "
              f"{np.median(at_best):.2f} ({min(at_best):.2f}~{max(at_best):.2f})")

In [ ]:
def fit_early(model, optimizer, patience=50, max_epochs=1000, seed=0):
    """조기 종료를 넣은 학습 루프. 멈춘 뒤 검증 손실이 가장 낮았던 가중치로 되돌린다."""
    shuffler = torch.Generator().manual_seed(seed)
    best, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(X_tr), generator=shuffler)
        for start in range(0, len(X_tr), 32):
            rows = order[start:start + 32]
            optimizer.zero_grad()
            loss_fn(model(X_tr[rows]), Z_tr[rows]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = loss_fn(model(X_va), Z_va).item()
        if valid_loss < best:                             # 나아졌으면 저장한다
            best, best_epoch = valid_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:              # 오래 나아지지 않으면 멈춘다
            break
    last_mark = rmse(Y25, gw(model, X25))
    model.load_state_dict(best_state)                 # 가장 좋았던 가중치로 되돌린다
    return epoch, best_epoch, last_mark, rmse(Y25, gw(model, X25))

model = make(0)
stop, best_epoch, last_mark, best_mark = fit_early(model, adam()(model.parameters()))
print(stop, best_epoch, f"{last_mark:.2f}", f"{best_mark:.2f}")
big_early = model                                         # 그림에서 쓴다

## 6.3 가중치 감쇠: L2 규제와 AdamW

In [ ]:
def shrink(make_optimizer, steps=100):
    """자료에서 오는 기울기가 0인 가중치 하나(처음 값 1)가 규제만으로 줄어드는 정도."""
    weight = torch.nn.Parameter(torch.tensor([1.0]))
    optimizer = make_optimizer([weight])
    for _ in range(steps):
        optimizer.zero_grad()
        (weight * 0.0).sum().backward()               # 자료에서 오는 기울기는 0이다
        optimizer.step()
    return round(weight.item(), 4)

for decay in [0.0001, 0.01, 1.0]:
    options = dict(lr=0.01, weight_decay=decay)
    plain = shrink(lambda params: torch.optim.SGD(params, **options))
    mixed = shrink(lambda params: torch.optim.Adam(params, **options))
    apart = shrink(lambda params: torch.optim.AdamW(params, **options))
    by_hand = (1 - 0.01 * decay) ** 100
    print(f"{decay:<7g} {plain:>8} {mixed:>8} {apart:>8}   {by_hand:.4f}")

In [ ]:
def size(model):
    """Linear 층의 가중치를 모두 제곱해서 더한 값. 편향은 넣지 않는다."""
    return sum((layer.weight ** 2).sum().item()
               for layer in model if isinstance(layer, nn.Linear))

def adamw(decay, rate=0.003):
    return lambda params: torch.optim.AdamW(params, lr=rate, weight_decay=decay)

print("Adam에 weight_decay를 준 경우(손실에 L2 규제를 더한 것과 같다)")
for decay in [0.0001, 0.001, 0.01, 0.1]:
    results[f"Adam+L2 {decay:g}"] = trial(make, adam(weight_decay=decay))
    show(f"L2 {decay:g}", results[f"Adam+L2 {decay:g}"])
print("AdamW")
for decay in [0.01, 0.1, 1.0, 3.0]:
    results[f"AdamW {decay:g}"] = trial(make, adamw(decay))
    show(f"감쇠 {decay:g}", results[f"AdamW {decay:g}"])

In [ ]:
kept = {"규제 없음": big_last}                     # 6.1절에서 학습시킨 모델
choices = [("Adam+L2 0.01", adam(weight_decay=0.01)),
           ("AdamW 0.01", adamw(0.01)), ("AdamW 3", adamw(3.0))]
for name, make_optimizer in choices:
    kept[name] = make(0)
    fit(kept[name], make_optimizer(kept[name].parameters()))
print("처음", round(size(make(0)), 1))
for name, net in kept.items():
    print(pad(name, 16) + f"{size(net):>6.1f}  " +
          " ".join(f"{v:>5.1f}" for v in gw(net, grid_x)))

## 6.4 드롭아웃: 학습할 때 유닛을 무작위로 끄기

In [ ]:
torch.manual_seed(0)
drop = nn.Dropout(0.5)
ones = torch.ones(10)
drop.train()
print(drop(ones).tolist())
print(round(torch.stack([drop(ones) for _ in range(10000)]).mean().item(), 3))
drop.eval()
print(drop(ones).tolist())

In [ ]:
for rate in [0.1, 0.3, 0.5]:
    results[f"드롭아웃 {rate}"] = trial(lambda seed: make(seed, drop=rate), adam())
    show(f"드롭아웃 {rate}", results[f"드롭아웃 {rate}"])

In [ ]:
model = make(0, drop=0.3)
fit(model, adam()(model.parameters()), epochs=300)
model.train()                                       # 학습 모드: 드롭아웃이 켜져 있다
torch.manual_seed(0)
with torch.no_grad():
    noisy = [loss_fn(model(X_tr), Z_tr).item() for _ in range(200)]
    noisy_valid = [loss_fn(model(X_va), Z_va).item() for _ in range(200)]
model.eval()                                        # 평가 모드: 드롭아웃이 꺼져 있다
with torch.no_grad():
    clean = loss_fn(model(X_tr), Z_tr).item()
    clean_valid = loss_fn(model(X_va), Z_va).item()
print("학습 자료:", round(np.mean(noisy), 4), round(min(noisy), 4),
      round(max(noisy), 4), round(clean, 4))
print("검증 자료:", round(np.mean(noisy_valid), 4), round(clean_valid, 4))

## 6.5 규제 방법 비교하기

In [ ]:
picked = {}

def row(name, logs, early):
    """early가 참이면 검증 손실이 가장 낮았던 에폭, 거짓이면 마지막 에폭의 결과."""
    picks = [log[log[:, 1].argmin()] if early else log[-1] for log in logs]
    valid, marks = np.array(picks)[:, 1], np.array(picks)[:, 2]
    picked[name] = marks
    print(pad(name, 36) + f"{np.median(valid):.3f}   {np.median(marks):.2f} "
          f"({marks.min():.2f}~{marks.max():.2f})")

print(pad("방법", 36) + "검증    2025년 RMSE")
row("큰 모델, 규제 없음", results["큰 모델"], False)
row("큰 모델 + 조기 종료", results["큰 모델"], True)
row("큰 모델 + AdamW 3", results["AdamW 3"], False)
row("큰 모델 + AdamW 3 + 조기 종료", results["AdamW 3"], True)
row("큰 모델 + L2 0.01", results["Adam+L2 0.01"], False)
row("큰 모델 + 드롭아웃 0.1", results["드롭아웃 0.1"], False)
row("큰 모델 + 드롭아웃 0.1 + 조기 종료", results["드롭아웃 0.1"], True)
row("작은 모델, 규제 없음", results["작은 모델"], False)
row("작은 모델 + 조기 종료", results["작은 모델"], True)

In [ ]:
def middle(logs, early=False):
    marks = [log[log[:, 1].argmin(), 2] if early else log[-1, 2] for log in logs]
    return np.array(marks)

resplit = {}
print("검증 주  검증 기온      큰 모델   +조기 종료  +AdamW 3  작은 모델")
for residue in range(4):
    tensors, (t_mean, t_std, y_mean, y_std), held = prepare(residue)
    X_tr, Z_tr, X_va, Z_va, X25 = tensors
    big = trial(make, adam(), epochs=400)
    decayed = trial(make, adamw(3.0), epochs=400)
    small = trial(lambda seed: make(seed, width=8, depth=1), adam(), epochs=400)
    resplit[residue] = [middle(big), middle(big, True), middle(decayed), middle(small)]
    span = f"{T23[held].min():.1f}~{T23[held].max():.1f}"
    print(f"{residue:<8} {span:<14} " +
          "      ".join(f"{np.median(marks):.2f}" for marks in resplit[residue]))

tensors, (t_mean, t_std, y_mean, y_std), held = prepare(1)   # 본문의 나누기로 되돌린다
X_tr, Z_tr, X_va, Z_va, X25 = tensors

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, torch.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 6-2
epochs = np.arange(1, 1001)
fig, axes = plt.subplots(2, 1, figsize=(7, 7.8))
axes[0].plot(epochs, big_log[:, 0], color=NAVY, lw=1.8, label="학습 자료 185일의 손실")
axes[0].plot(epochs, big_log[:, 1], color=BRICK, lw=1.4, label="검증 자료 62일의 손실")
axes[0].scatter([turn + 1], [big_log[turn, 1]], s=46, color=BRICK, zorder=5, edgecolor="white", lw=0.8)
axes[0].annotate(f"검증 손실이 가장 낮은 곳\n({turn + 1}에폭, {big_log[turn, 1]:.3f})", (turn + 1, big_log[turn, 1]), (1.25, 0.152),
                 fontsize=9, color=BRICK, va="center", linespacing=1.4, arrowprops=dict(arrowstyle="-", color=BRICK, lw=0.7))
axes[0].set_ylim(0.14, 0.30); axes[0].set_ylabel("손실 (표준화한 정답의 평균 제곱 오차)", fontsize=9)
axes[1].plot(epochs, big_log[:, 2], color=BRICK, lw=1.4, label=f"큰 모델 (가중치와 편향 8,513개): 1,000에폭 뒤 {big_log[-1, 2]:.2f}")
axes[1].plot(epochs, small_log[:, 2], color=TEAL, lw=1.8, label=f"작은 모델 (25개): 1,000에폭 뒤 {small_log[-1, 2]:.2f}")
axes[1].set_ylim(2.9, 4.0); axes[1].set_ylabel("2025년 RMSE (GW)", fontsize=9)
for ax in axes:
    ax.set_xscale("log"); ax.set_xlim(1, 1000); plain_ticks(ax.xaxis); ax.minorticks_off()
    ax.set_xlabel("에폭 (로그 축, 시드 0)", fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper right" if ax is axes[1] else "upper center")
    frame(ax, "① 큰 모델의 학습 손실과 검증 손실" if ax is axes[0] else "② 두 모델의 2025년 RMSE")
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_6_2.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-3
wide = np.linspace(-11, 34, 600)
wide_x = column((wide - t_mean) / t_std)
fig, ax = plt.subplots(figsize=(7, 5.4))
ax.scatter(T_tr, Y_tr, s=9, color="#c3c9d1", label="학습 자료 185일", zorder=1)
ax.scatter(T_va, Y_va, s=13, facecolor="white", edgecolor=NAVY, lw=0.7, label="검증 자료 62일", zorder=2)
ax.plot(wide, gw(big_last, wide_x), color=BRICK, lw=1.6, label="큰 모델, 1,000에폭", zorder=4)
ax.plot(wide, gw(big_best, wide_x), color=NAVY, lw=1.6, ls=(0, (5, 2)), label=f"큰 모델, {turn + 1}에폭 (검증 손실이 가장 낮은 곳)", zorder=3)
ax.plot(wide, gw(small_last, wide_x), color=TEAL, lw=2.2, label="작은 모델, 1,000에폭", zorder=3)
for edge in (T_tr.min(), T_tr.max()):
    ax.axvline(edge, color=GRAY, lw=0.8, ls=(0, (2, 2)))
ax.text(T_tr.min() + 0.4, 91.5, "학습 자료의 기온 범위", fontsize=9, color="#5b6470", va="top")
ax.set_xlim(-11, 34); ax.set_ylim(52, 92)
ax.set_xlabel("하루 평균 기온 (℃)", fontsize=9); ax.set_ylabel("하루 평균 전력수요 (GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-6
looks = {"규제 없음": (BRICK, "-", 1.5), "AdamW 0.01": (GRAY, (0, (5, 2)), 1.5), "Adam+L2 0.01": (TEAL, "-", 2.2), "AdamW 3": (NAVY, "-", 1.7)}
fig, axes = plt.subplots(2, 1, figsize=(7, 8.2), gridspec_kw={"height_ratios": [2.2, 1]})
axes[0].scatter(T_tr, Y_tr, s=9, color="#c3c9d1", zorder=1)
for name, (col, ls, lw) in looks.items():
    axes[0].plot(wide, gw(kept[name], wide_x), color=col, ls=ls, lw=lw, label=name)
axes[0].set_xlim(-11, 34); axes[0].set_ylim(52, 92)
axes[0].set_xlabel("하루 평균 기온 (℃)", fontsize=9); axes[0].set_ylabel("하루 평균 전력수요 (GW)", fontsize=9)
axes[0].legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
frame(axes[0], "① 학습을 마친 모델의 예측")
names = list(looks)[::-1]
axes[1].barh(names, [size(kept[name]) for name in names], color=[looks[name][0] for name in names], height=0.56)
for k, name in enumerate(names):
    axes[1].text(size(kept[name]) * 1.08, k, f"{size(kept[name]):.1f}", fontsize=9, color=looks[name][0], va="center", weight="bold")
axes[1].axvline(size(make(0)), color="#5b6470", lw=0.9, ls=(0, (2, 2)))
axes[1].text(size(make(0)) * 1.06, 3.62, "학습 전: 63.4", fontsize=9, color="#5b6470", va="center")
axes[1].set_xscale("log"); axes[1].set_xlim(5, 1500); axes[1].set_ylim(-0.6, 3.9); plain_ticks(axes[1].xaxis); axes[1].minorticks_off()
axes[1].set_xlabel("가중치를 제곱해서 더한 값 (로그 축, 시드 0, 1,000에폭 뒤)", fontsize=9)
frame(axes[1], "② 가중치의 크기", grid="x")
fig.tight_layout(h_pad=2.0)
fig.savefig("fig_6_6.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-8
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.hist(noisy, bins=24, color="#e6c3b1", edgecolor="white", lw=0.5)
marks = [(np.mean(noisy), BRICK, "학습 모드로 잰 학습 손실의 평균", 0.97), (clean, NAVY, "평가 모드로 잰 학습 손실", 0.80),
         (clean_valid, TEAL, "평가 모드로 잰 검증 손실", 0.97)]
top = ax.get_ylim()[1] * 1.22
for value, col, name, height in marks:
    ax.axvline(value, color=col, lw=1.6)
    side = -1 if col == BRICK else 1
    ax.text(value + side * 0.0012, top * height, f"{name}\n{value:.3f}", fontsize=9, color=col, va="top",
            ha="right" if side < 0 else "left", bbox=BOX)
ax.set_ylim(0, top); ax.set_xlim(0.19, 0.30)
ax.set_xlabel("손실 (같은 모델, 같은 학습 자료 185일을 학습 모드로 200번 잰 값의 분포)", fontsize=9)
ax.set_ylabel("횟수", fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-9
names = list(picked)[::-1]
fig, ax = plt.subplots(figsize=(7, 5.2))
for k, name in enumerate(names):
    col = TEAL if name.startswith("작은") else (BRICK if name == "큰 모델, 규제 없음" else NAVY)
    marks = picked[name]
    ax.plot([marks.min(), marks.max()], [k, k], color=col, lw=1.1)
    ax.scatter(marks, [k] * 5, s=20, facecolor="white", edgecolor=col, lw=0.9, zorder=4)
    ax.scatter([np.median(marks)], [k], s=64, color=col, marker="D", zorder=5, edgecolor="white", lw=0.8)
    ax.text(3.68, k, f"{np.median(marks):.2f}", fontsize=9, color=col, va="center", weight="bold")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
ax.set_xlim(3.0, 3.75); ax.set_ylim(-0.6, len(names) - 0.4)
ax.set_xlabel("2025년 RMSE (GW). 빈 동그라미는 시드 5개, 마름모는 중앙값", fontsize=9)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_6_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 6-10
looks = [("큰 모델, 400에폭", BRICK, "o"), ("큰 모델 + 조기 종료", NAVY, "s"), ("큰 모델 + AdamW 3", "#5b6470", "^"), ("작은 모델, 400에폭", TEAL, "D")]
fig, ax = plt.subplots(figsize=(7, 4.8))
for residue in range(4):
    for k, (name, col, marker) in enumerate(looks):
        marks = resplit[residue][k]
        x = residue + (k - 1.5) * 0.17
        ax.plot([x, x], [marks.min(), marks.max()], color=col, lw=1.2)
        ax.text(x, marks.max() + 0.025, f"{np.median(marks):.2f}", fontsize=9, color=col, ha="center", va="bottom", rotation=90, weight="bold")
        ax.scatter([x], [np.median(marks)], s=44, color=col, marker=marker, zorder=5, edgecolor="white", lw=0.7,
                   label=name if residue == 0 else None)
ax.set_xticks(range(4)); ax.set_xticklabels([f"나머지 {residue}인 주를\n검증 자료로" for residue in range(4)], fontsize=9)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(2.8, 4.35)
ax.set_ylabel("2025년 RMSE (GW). 점은 시드 5개의 중앙값, 세로선은 범위", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
frame(ax)
fig.tight_layout()
fig.savefig("fig_6_10.png", dpi=300)
plt.show()